# CYGNSS L1 coherency-screening OmF figures

Map and monthly-summary figures for the three CYGNSS L1 DA experiment comparisons staged in `output/coherency_screening_stats`.


In [ ]:
from __future__ import annotations

import calendar
import math
import pickle
import struct
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize, TwoSlopeNorm
from netCDF4 import Dataset
from IPython.display import display

REPO = Path.cwd()
if REPO.name != "geosldas-analysis":
    REPO = Path("/Users/amfox/Desktop/geosldas-analysis")
PROJECT = REPO / "projects" / "CYGNSS_L1_AZ"
STATS = PROJECT / "output" / "coherency_screening_stats"
OUT = PROJECT / "output" / "coherency_screening_figures"
TILECOORD = PROJECT / "OLv8_M36_all_sensors_AZ_describe" / "OLv8_M36_all_sensors_AZ.ldas_tilecoord.bin"
OUT.mkdir(parents=True, exist_ok=True)

NMIN = 10
MAP_EXTENT = (-118.6, -105.2, 28.6, 40.4)
PCT_LIMIT = 30.0

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})

In [ ]:
@dataclass(frozen=True)
class Group:
    name: str
    indices: tuple[int, ...]
    units: str
    color: str


GROUPS = (
    Group("CYGL1", (12,), "dB", "#7b3294"),
    Group("CYGL3", (11,), "m$^3$ m$^{-3}$", "#d95f02"),
    Group("ASCAT", (8, 9, 10), "m$^3$ m$^{-3}$", "#1b9e77"),
    Group("SMOS", (0, 1, 2, 3), "K", "#7570b3"),
    Group("SMAP", (4, 5, 6, 7), "K", "#666666"),
)


@dataclass(frozen=True)
class Experiment:
    key: str
    label: str
    ol_nc4: Path
    da_nc4: Path
    ol_pkl: Path
    da_pkl: Path
    ol_latmask_pkl: Path
    da_latmask_pkl: Path


EXPERIMENTS = (
    Experiment(
        "gated_dense",
        "Gated dense",
        STATS / "temporal_stats_OL_paired_monitor_xmask_gated_dense_20200101_20201231.nc4",
        STATS / "temporal_stats_DA_gated_dense_20200101_20201231.nc4",
        STATS / "spatial_stats_OL_paired_monitor_xmask_gated_dense_202001_202012.pkl",
        STATS / "spatial_stats_DA_gated_dense_202001_202012.pkl",
        STATS / "spatial_stats_OL_paired_monitor_xmask_gated_dense_latmask37p5_202001_202012.pkl",
        STATS / "spatial_stats_DA_gated_dense_latmask37p5_202001_202012.pkl",
    ),
    Experiment(
        "coherency_screened",
        "Coherency screened",
        STATS / "temporal_stats_OL_paired_monitor_xmask_coherency_A_20200101_20201231.nc4",
        STATS / "temporal_stats_DA_coherency_screened_20200101_20201231.nc4",
        STATS / "spatial_stats_OL_paired_monitor_xmask_coherency_A_202001_202012.pkl",
        STATS / "spatial_stats_DA_coherency_screened_202001_202012.pkl",
        STATS / "spatial_stats_OL_paired_monitor_xmask_coherency_A_latmask37p5_202001_202012.pkl",
        STATS / "spatial_stats_DA_coherency_screened_latmask37p5_202001_202012.pkl",
    ),
    Experiment(
        "coherency_randmatch",
        "Random matched",
        STATS / "temporal_stats_OL_paired_monitor_xmask_coherency_B_20200101_20201231.nc4",
        STATS / "temporal_stats_DA_coherency_randmatch_20200101_20201231.nc4",
        STATS / "spatial_stats_OL_paired_monitor_xmask_coherency_B_202001_202012.pkl",
        STATS / "spatial_stats_DA_coherency_randmatch_202001_202012.pkl",
        STATS / "spatial_stats_OL_paired_monitor_xmask_coherency_B_latmask37p5_202001_202012.pkl",
        STATS / "spatial_stats_DA_coherency_randmatch_latmask37p5_202001_202012.pkl",
    ),
)


In [ ]:
def read_exact(stream, nbytes: int) -> bytes:
    data = stream.read(nbytes)
    if len(data) != nbytes:
        raise EOFError(f"expected {nbytes} bytes, got {len(data)}")
    return data


def read_tilecoord(path: Path) -> dict[str, np.ndarray]:
    int_fields = {"tile_id", "typ", "pfaf", "i_indg", "j_indg"}
    fields = [
        "tile_id", "typ", "pfaf", "com_lon", "com_lat", "min_lon", "max_lon",
        "min_lat", "max_lat", "i_indg", "j_indg", "frac_cell", "frac_pfaf", "area", "elev",
    ]
    out = {}
    with path.open("rb") as stream:
        tag = struct.unpack("<i", read_exact(stream, 4))[0]
        if tag != 4:
            raise ValueError(f"unexpected N_tile record tag {tag}")
        n_tile = struct.unpack("<i", read_exact(stream, 4))[0]
        if struct.unpack("<i", read_exact(stream, 4))[0] != tag:
            raise ValueError("N_tile record tags do not match")
        out["N_tile"] = np.asarray(n_tile, dtype=np.int32)
        for field in fields:
            dtype = np.dtype("<i4") if field in int_fields else np.dtype("<f4")
            expected = n_tile * dtype.itemsize
            tag = struct.unpack("<i", read_exact(stream, 4))[0]
            if tag != expected:
                raise ValueError(f"unexpected record tag for {field}: {tag}, expected {expected}")
            out[field] = np.frombuffer(read_exact(stream, expected), dtype=dtype).copy()
            if struct.unpack("<i", read_exact(stream, 4))[0] != tag:
                raise ValueError(f"record tags do not match for {field}")
    return out


def read_nc(path: Path, names=("OmF_stdv", "OmF_mean", "N_data")) -> dict[str, np.ndarray]:
    with Dataset(path) as ds:
        return {
            name: np.ma.filled(ds.variables[name][:], np.nan).astype(float)
            for name in names
        }


def read_monthly(path: Path) -> dict[str, np.ndarray]:
    with path.open("rb") as stream:
        raw = pickle.load(stream)
    source = raw.get("monthly", raw) if isinstance(raw, dict) else raw
    data = {
        key: np.asarray(value, dtype=float) if key != "date_vec" else value
        for key, value in source.items()
    }
    if "date_vec" in data:
        months = []
        for value in data["date_vec"]:
            if isinstance(value, datetime):
                months.append(value)
            elif isinstance(value, np.datetime64):
                months.append(value.astype("datetime64[D]").astype(datetime))
            else:
                months.append(datetime.strptime(str(value)[:6], "%Y%m"))
        data["months"] = np.asarray(months)
    else:
        data["months"] = np.asarray([datetime(2020, month, 1) for month in range(1, data["N_data"].shape[0] + 1)])
    if isinstance(raw, dict) and "mask_info" in raw:
        data["mask_info"] = raw["mask_info"]
    return data


def weighted_species(values: np.ndarray, counts: np.ndarray, group: Group, nmin: int = NMIN) -> np.ndarray:
    idx = list(group.indices)
    group_values = values[..., idx]
    group_counts = counts[..., idx]
    valid = np.isfinite(group_values) & np.isfinite(group_counts) & (group_counts >= nmin)
    numerator = np.where(valid, group_values * group_counts, 0.0).sum(axis=-1)
    denominator = np.where(valid, group_counts, 0.0).sum(axis=-1)
    return np.divide(numerator, denominator, out=np.full(denominator.shape, np.nan), where=denominator > 0)


def group_series(data: dict[str, np.ndarray], variable: str, group: Group) -> np.ndarray:
    return weighted_species(data[variable], data["N_data"], group)


def percent_diff(da: np.ndarray, ol: np.ndarray) -> np.ndarray:
    return 100.0 * (da - ol) / ol


def norm_from_values(values, lower=2, upper=98):
    finite = np.concatenate([np.ravel(v[np.isfinite(v)]) for v in values if np.any(np.isfinite(v))])
    if finite.size == 0:
        return Normalize(0, 1)
    vmin, vmax = np.nanpercentile(finite, (lower, upper))
    if math.isclose(vmin, vmax):
        delta = max(abs(vmin) * 0.05, 1.0e-6)
        vmin -= delta
        vmax += delta
    return Normalize(vmin=vmin, vmax=vmax)


def signed_norm(values, limit=None, percentile=98):
    if limit is None:
        finite = np.concatenate([np.ravel(np.abs(v[np.isfinite(v)])) for v in values if np.any(np.isfinite(v))])
        limit = float(np.nanpercentile(finite, percentile)) if finite.size else 1.0
    limit = max(float(limit), 1.0e-12)
    return TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit)


In [ ]:
tilecoord = read_tilecoord(TILECOORD)
lon = np.asarray(tilecoord["com_lon"], dtype=float)
lat = np.asarray(tilecoord["com_lat"], dtype=float)

spatial = {exp.key: {"ol": read_nc(exp.ol_nc4), "da": read_nc(exp.da_nc4)} for exp in EXPERIMENTS}
monthly_unmasked = {exp.key: {"ol": read_monthly(exp.ol_pkl), "da": read_monthly(exp.da_pkl)} for exp in EXPERIMENTS}
monthly_latmask = {exp.key: {"ol": read_monthly(exp.ol_latmask_pkl), "da": read_monthly(exp.da_latmask_pkl)} for exp in EXPERIMENTS}

for exp in EXPERIMENTS:
    mask_info = monthly_latmask[exp.key]["ol"].get("mask_info", {})
    print(exp.label)
    print("  map arrays:", spatial[exp.key]["ol"]["OmF_stdv"].shape)
    print("  monthly arrays:", monthly_unmasked[exp.key]["ol"]["OmF_stdv"].shape)
    print("  latmask monthly arrays:", monthly_latmask[exp.key]["ol"]["OmF_stdv"].shape, mask_info)

for variable in ("OmF_stdv", "OmF_mean"):
    print(f"\nMax |latmask - unmasked| for CYGNSS monthly {variable}")
    for exp in EXPERIMENTS:
        for side in ("ol", "da"):
            for group in GROUPS[:2]:
                old = group_series(monthly_unmasked[exp.key][side], variable, group)
                new = group_series(monthly_latmask[exp.key][side], variable, group)
                print(f"  {exp.label:19s} {side.upper():2s} {group.name:5s}: {np.nanmax(np.abs(new - old)):.3e}")


In [ ]:
def base_map(ax):
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.94", zorder=0)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), facecolor="white", zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.45, edgecolor="0.25")
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.35, edgecolor="0.35")
    states = cfeature.NaturalEarthFeature(
        "cultural", "admin_1_states_provinces_lakes", "50m", facecolor="none"
    )
    ax.add_feature(states, lw=0.35, edgecolor="0.45")


def scatter_tiles(ax, values, cmap, norm):
    valid = np.isfinite(values)
    return ax.scatter(
        lon[valid], lat[valid], c=values[valid], s=19, marker="s", linewidths=0,
        cmap=cmap, norm=norm, transform=ccrs.PlateCarree(), zorder=2,
    )


def plot_map_set(exp: Experiment):
    ol_data = spatial[exp.key]["ol"]
    da_data = spatial[exp.key]["da"]
    rows = []
    for group in GROUPS:
        ol = group_series(ol_data, "OmF_stdv", group)
        da = group_series(da_data, "OmF_stdv", group)
        rows.append((group, ol, da, percent_diff(da, ol)))

    fig = plt.figure(figsize=(13.2, 14.8))
    percent_norm = signed_norm([row[3] for row in rows], limit=PCT_LIMIT)
    image_by_col = {}
    for row_index, (group, ol, da, pct) in enumerate(rows):
        value_norm = norm_from_values([ol, da])
        for col_index, (values, title, cmap, norm) in enumerate((
            (ol, "OL OmF StDev", "viridis", value_norm),
            (da, "DA OmF StDev", "viridis", value_norm),
            (pct, "(DA - OL) / OL (%)", "RdBu_r", percent_norm),
        )):
            ax = fig.add_subplot(5, 3, row_index * 3 + col_index + 1, projection=ccrs.PlateCarree())
            base_map(ax)
            image = scatter_tiles(ax, values, cmap, norm)
            image_by_col[col_index] = image
            if row_index == 0:
                ax.set_title(title, fontweight="bold", pad=8)
            if col_index == 0:
                ax.text(-0.07, 0.5, group.name, transform=ax.transAxes, rotation=90,
                        va="center", ha="center", fontweight="bold", fontsize=11)
            ax.set_xticks([])
            ax.set_yticks([])
            label = "%" if col_index == 2 else group.units
            cb = fig.colorbar(image, ax=ax, orientation="horizontal", fraction=0.046, pad=0.035)
            cb.set_label(label, fontsize=8)
            cb.ax.tick_params(labelsize=7)

    fig.suptitle(f"{exp.label}: OmF standard deviation maps, January-December 2020", y=0.995, fontsize=14)
    fig.subplots_adjust(top=0.965, bottom=0.025, left=0.055, right=0.985, hspace=0.25, wspace=0.08)
    path = OUT / f"{exp.key}_omf_stdv_maps_5x3.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    return fig, path

map_paths = []
for exp in EXPERIMENTS:
    fig, path = plot_map_set(exp)
    map_paths.append(path)
    plt.show()
    plt.close(fig)

[path.relative_to(PROJECT) for path in map_paths]


In [ ]:
def aggregate_monthly(data: dict[str, np.ndarray], variable: str, group: Group) -> np.ndarray:
    return group_series(data, variable, group)


def setup_month_axis(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax.grid(True, lw=0.4, alpha=0.35)


def plot_monthly_set(variable: str, figure_label: str, monthly_data, suffix: str, title_note: str):
    months = monthly_data[EXPERIMENTS[0].key]["ol"]["months"]
    fig, axes = plt.subplots(5, 2, figsize=(13.0, 12.6), sharex=True)

    for row_index, group in enumerate(GROUPS):
        left = axes[row_index, 0]
        right = axes[row_index, 1]

        colors = {
            "gated_dense": "#1f77b4",
            "coherency_screened": "#d62728",
            "coherency_randmatch": "#2ca02c",
        }
        for exp in EXPERIMENTS:
            ol = aggregate_monthly(monthly_data[exp.key]["ol"], variable, group)
            da = aggregate_monthly(monthly_data[exp.key]["da"], variable, group)
            left.plot(months, ol, color="0.35", lw=1.15, ls="--", alpha=0.75,
                      label="matching OL" if row_index == 0 and exp.key == EXPERIMENTS[0].key else None)
            color = colors[exp.key]
            left.plot(months, da, color=color, lw=1.9, label=exp.label if row_index == 0 else None)
            right.plot(months, percent_diff(da, ol), color=color, lw=1.9, label=exp.label if row_index == 0 else None)

        left.set_ylabel(f"{group.name}\n{group.units}", fontweight="bold")
        right.axhline(0, color="0.25", lw=0.8)
        right.set_ylabel("%")
        setup_month_axis(left)
        setup_month_axis(right)
        if row_index == 0:
            left.set_title(f"{figure_label}: matching OL baselines and DA experiments", fontweight="bold")
            right.set_title("Percent difference from matching OL", fontweight="bold")
        if row_index == len(GROUPS) - 1:
            left.set_xlabel("2020")
            right.set_xlabel("2020")

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.985))
    fig.suptitle(f"Monthly {figure_label}, January-December 2020\n{title_note}", y=1.025, fontsize=14)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    path = OUT / f"monthly_{variable.lower()}_{suffix}_5x2.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    return fig, path

monthly_paths = []
for variable, label in (("OmF_stdv", "OmF StDev"), ("OmF_mean", "OmF mean")):
    fig, path = plot_monthly_set(
        variable,
        label,
        monthly_latmask,
        "latmask37p5",
        "Lat-masked summaries: tile center latitude < 37.5 deg N",
    )
    monthly_paths.append(path)
    plt.show()
    plt.close(fig)

[path.relative_to(PROJECT) for path in monthly_paths]


In [ ]:
# Report figure: full-period outcome of the coherency-screening experiment.
def read_full_period_groups(path: Path) -> dict[str, dict[str, float]]:
    with path.open('rb') as stream:
        data = pickle.load(stream)
    return data['full_period']['per_group']

rows = []
for exp in EXPERIMENTS:
    ol_groups = read_full_period_groups(exp.ol_latmask_pkl)
    da_groups = read_full_period_groups(exp.da_latmask_pkl)
    for group_key, label in (
        ('CygL1 (dB)', 'CygL1\nown obs'),
        ('Tb (K)', 'Tb\nverification'),
        ('SM (m3/m3)', 'SM\nverification'),
    ):
        ol_stdv = ol_groups[group_key]['OmF_stdv']
        da_stdv = da_groups[group_key]['OmF_stdv']
        rows.append({
            'experiment': exp.key,
            'experiment_label': exp.label,
            'group': label,
            'percent': 100.0 * (da_stdv - ol_stdv) / ol_stdv,
            'ol_stdv': ol_stdv,
            'da_stdv': da_stdv,
            'n': da_groups[group_key]['N'],
        })

summary = pd.DataFrame(rows)
display(summary.pivot(index='group', columns='experiment_label', values='percent').round(2))

colors = {
    'Gated dense': '#1f77b4',
    'Coherency screened': '#d62728',
    'Random matched': '#2ca02c',
}
fig, ax = plt.subplots(figsize=(8.2, 4.2))
groups = ['CygL1\nown obs', 'Tb\nverification', 'SM\nverification']
experiments = [exp.label for exp in EXPERIMENTS]
x = np.arange(len(groups))
width = 0.24
for offset, exp_label in zip((-width, 0, width), experiments):
    vals = [summary[(summary.group == g) & (summary.experiment_label == exp_label)].percent.iloc[0] for g in groups]
    ax.bar(x + offset, vals, width, label=exp_label, color=colors[exp_label])
    for xi, yi in zip(x + offset, vals):
        va = 'bottom' if yi >= 0 else 'top'
        dy = 0.18 if yi >= 0 else -0.18
        ax.text(xi, yi + dy, f'{yi:+.1f}%', ha='center', va=va, fontsize=8)

ax.axhline(0, color='0.25', lw=0.9)
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.set_ylabel('OmF StDev change from matching OL (%)')
ax.set_title('Coherency screening improves CygL1 fit, but not independent verification', loc='left', fontweight='bold')
ax.text(0.01, 0.03, 'Negative = lower OmF variability than matching OL; summaries use tile center latitude < 37.5 deg N.',
        transform=ax.transAxes, fontsize=8.5, color='0.25')
ax.legend(frameon=False, ncol=3, loc='upper right')
ax.set_ylim(min(-10, summary.percent.min() - 1.5), max(4, summary.percent.max() + 1.5))
fig.tight_layout()
path = OUT / 'report_coherency_screening_full_period_outcome.png'
fig.savefig(path, dpi=220, bbox_inches='tight')
path.relative_to(PROJECT)


Saved figures:

- `output/coherency_screening_figures/gated_dense_omf_stdv_maps_5x3.png`
- `output/coherency_screening_figures/coherency_screened_omf_stdv_maps_5x3.png`
- `output/coherency_screening_figures/coherency_randmatch_omf_stdv_maps_5x3.png`
- `output/coherency_screening_figures/monthly_omf_stdv_latmask37p5_5x2.png`
- `output/coherency_screening_figures/monthly_omf_mean_latmask37p5_5x2.png`
